# Step 7b: Prepare the impervious layer

Create `crest_IM.tif` from the GMIS 2010 data for the Ghana example.
Values represent impervious surface in percent (0–100). The output keeps
the source's 250 m pixel size; the DEM defines the model area.

Run from `Example/Example_notebooks` with the **EF5 Builder** kernel.
Downloads require an Earthdata Login token. [Data source and method](../../docs/impervious.md).


In [ ]:
import json
import os
import re
from contextlib import ExitStack
from getpass import getpass
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlparse
from zipfile import ZipFile, is_zipfile

import numpy as np
import requests
import rasterio
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window, from_bounds
from scipy.ndimage import distance_transform_edt


## 1. Paths and settings

Check the paths. Set `overwrite=True` to replace processed outputs.
Previously downloaded ZIP files are reused.


In [ ]:
dem_file = Path("../data_model/basic/dem.tif")
download_dir = Path("../useful_datasets/impervious/gmis_2010_250m")
output_dir = Path("../data_model/crest_parameters")
output_file = output_dir / "crest_IM.tif"
diagnostics_dir = Path("../posproc_outputs/crest_diagnostics")
fill_mask_file = diagnostics_dir / "crest_IM_filled_mask.tif"
overwrite = False
fill_missing = True  # Nearest-neighbor estimates for missing cells inside the DEM.

if not dem_file.is_file():
    raise FileNotFoundError(dem_file)
for path in (output_file, fill_mask_file):
    if path.exists() and not overwrite:
        raise FileExistsError(f"{path} already exists; set overwrite=True to replace processed outputs.")
with rasterio.open(dem_file) as dem:
    if dem.count != 1 or dem.crs != rasterio.crs.CRS.from_epsg(4326):
        raise ValueError("This example requires a single-band EPSG:4326 DEM.")
    dem_bounds = tuple(dem.bounds)
    print("DEM bounds (west, south, east, north):", dem_bounds)
    print("DEM pixel size (degrees):", dem.res)
download_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
diagnostics_dir.mkdir(parents=True, exist_ok=True)


## 2. Find the GMIS tiles

Run the cell to list the impervious-percentage tiles covering the DEM.


In [ ]:
def get_json(url, params):
    response = requests.get(url, params=params, timeout=(15, 120))
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise RuntimeError(f"NASA service error: {data['error']}")
    return data

bbox = ",".join(str(value) for value in dem_bounds)
tile_index_url = "https://gis.earthdata.nasa.gov/portal/rest/services/Hosted/UTM_Tiles/FeatureServer/0/query"
tile_data = get_json(tile_index_url, {
    "f": "json", "where": "1=1", "geometry": bbox,
    "geometryType": "esriGeometryEnvelope", "inSR": 4326,
    "spatialRel": "esriSpatialRelIntersects", "outFields": "gzd", "returnGeometry": "false",
})
if tile_data.get("exceededTransferLimit"):
    raise RuntimeError("The tile-index response is incomplete.")
tile_ids = {feature["attributes"]["gzd"] for feature in tile_data.get("features", [])}
if not tile_ids:
    raise ValueError("No GMIS tiles intersect the DEM bounds.")

cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"
selected = {}
page = 1
while True:
    data = get_json(cmr_url, {
        "collection_concept_id": "C3550185860-ESDIS", "bounding_box": bbox,
        "page_size": 2000, "page_num": page,
    })
    entries = data.get("feed", {}).get("entry", [])
    for entry in entries:
        title = entry["title"]
        match = re.fullmatch(r"([0-9]{2}[A-Z])_gmis_impervious_surface_percentage_geographic_250m[.]zip", title)
        if not match or match.group(1) not in tile_ids:
            continue
        urls = [link["href"] for link in entry.get("links", [])
                if not link.get("inherited", False)
                and urlparse(link.get("href", "")).path.endswith("/" + title)]
        if len(urls) != 1:
            raise ValueError(f"Expected one data URL for {title}; got {urls}")
        selected[match.group(1)] = {"tile": match.group(1), "filename": title, "url": urls[0]}
    if tile_ids.issubset(selected) or len(entries) < 2000:
        break
    page += 1
if tile_ids - selected.keys():
    raise FileNotFoundError(f"Missing 250 m percentage tiles in CMR: {sorted(tile_ids-selected.keys())}")
manifest = [selected[key] for key in sorted(tile_ids)]
(download_dir / "download_manifest.json").write_text(json.dumps({
    "dataset": "GMIS 2010 v1", "collection_id": "C3550185860-ESDIS",
    "dem_bounds": dem_bounds, "tiles": manifest,
}, indent=2))
for item in manifest:
    print(item["tile"], item["url"])


## 3. Download the files

Generate a token at [Earthdata Login](https://urs.earthdata.nasa.gov/profile)
and enter it in the hidden prompt. Keep the token private.
You can also place manually downloaded ZIP files in `download_dir`.


In [ ]:
pending = []
for item in manifest:
    path = download_dir / item["filename"]
    if path.exists():
        if not is_zipfile(path):
            raise ValueError(f"Existing file is not a valid ZIP: {path}")
        print("Using existing:", path.name)
    else:
        pending.append(item)

if pending:
    token = os.environ.get("EARTHDATA_TOKEN") or getpass("Earthdata token (hidden): ")
    if not token.strip():
        raise ValueError("An Earthdata token is required for missing downloads.")
    try:
        with requests.Session() as session:
            for item in pending:
                url = item["url"]
                if urlparse(url).scheme != "https" or urlparse(url).hostname != "data.earthdata.nasa.gov":
                    raise ValueError(f"Unexpected data host in CMR: {url}")
                with TemporaryDirectory(prefix=".gmis-download-", dir=download_dir) as temp:
                    staged = Path(temp) / item["filename"]
                    with session.get(url, headers={"Authorization": f"Bearer {token}"},
                                     stream=True, timeout=(15, 120)) as response:
                        if response.status_code in (401, 403):
                            raise PermissionError("Earthdata denied access. Check your token and application permissions, or download the ZIP manually.")
                        response.raise_for_status()
                        with staged.open("wb") as dst:
                            for chunk in response.iter_content(chunk_size=1024*1024):
                                if chunk:
                                    dst.write(chunk)
                    if not is_zipfile(staged):
                        raise ValueError("The response was not a ZIP archive; it may be a login page.")
                    with ZipFile(staged) as archive:
                        if archive.testzip() is not None:
                            raise ValueError("The downloaded ZIP failed its integrity check.")
                    staged.replace(download_dir / item["filename"])
                    print("Downloaded:", item["filename"])
    finally:
        del token
zip_files = [download_dir / item["filename"] for item in manifest]


## 4. Clean and crop the tiles

Prepare the part of each tile covering the DEM. The cell handles GMIS missing-value
codes. [Source codes](../../docs/impervious.md).


In [ ]:
normalized_dir = download_dir / "cleaned_250m"
normalized_dir.mkdir(exist_ok=True)
normalized_files = []
source_resolution = None

for archive_path in zip_files:
    with ZipFile(archive_path) as archive:
        tiffs = [name for name in archive.namelist()
                 if name.lower().endswith((".tif", ".tiff"))]
    if len(tiffs) != 1:
        raise ValueError(f"Expected one percentage GeoTIFF in {archive_path.name}: {tiffs}")
    source_path = f"zip://{archive_path.resolve()}!{tiffs[0]}"
    normalized = normalized_dir / (archive_path.stem + ".tif")
    if normalized.exists() and not overwrite:
        raise FileExistsError(f"{normalized} already exists; set overwrite=True to recreate processed files.")
    with rasterio.open(source_path) as src:
        t = src.transform
        if src.crs != rasterio.crs.CRS.from_epsg(4326) or t.b or t.d or t.a <= 0 or t.e >= 0:
            raise ValueError("Expected a north-up geographic GMIS tile.")
        if src.count not in (1, 2):
            raise ValueError("Unexpected number of GMIS bands; check the source product.")
        if source_resolution is None:
            source_resolution = src.res
        elif not np.allclose(src.res, source_resolution, rtol=1e-6, atol=1e-12):
            raise ValueError("Source tiles have different pixel sizes; no automatic resolution change is allowed.")
        overlap = (max(dem_bounds[0], src.bounds.left), max(dem_bounds[1], src.bounds.bottom),
                   min(dem_bounds[2], src.bounds.right), min(dem_bounds[3], src.bounds.top))
        if overlap[0] >= overlap[2] or overlap[1] >= overlap[3]:
            continue
        crop = from_bounds(*overlap, transform=src.transform)
        left, top = max(0, int(np.floor(crop.col_off))), max(0, int(np.floor(crop.row_off)))
        right = min(src.width, int(np.ceil(crop.col_off+crop.width)))
        bottom = min(src.height, int(np.ceil(crop.row_off+crop.height)))
        crop = Window(left, top, right-left, bottom-top)
        profile = dict(driver="GTiff", count=1, width=int(crop.width), height=int(crop.height),
                       crs=src.crs, transform=src.window_transform(crop), dtype="float32",
                       nodata=-9999, tiled=False, interleave="pixel", compress="deflate", BIGTIFF="IF_SAFER")
        with TemporaryDirectory(prefix=".gmis-clean-", dir=normalized_dir) as temp:
            staged = Path(temp) / "cleaned.tif"
            with rasterio.open(staged, "w", **profile) as dst:
                for row in range(0, int(crop.height), 512):
                    for col in range(0, int(crop.width), 512):
                        block = Window(col, row, min(512, int(crop.width)-col), min(512, int(crop.height)-row))
                        read_window = Window(left+col, top+row, block.width, block.height)
                        values = src.read(1, window=read_window).astype("float32")
                        missing = ~np.isfinite(values) | (values == 255)
                        if src.nodata is not None and src.nodata not in (0, 200):
                            missing |= values == src.nodata
                        known = missing | ((values >= 0) & (values <= 100)) | (values == 200)
                        if not known.all():
                            raise ValueError(f"Unexpected GMIS codes: {np.unique(values[~known])[:10]}")
                        values[values == 200] = 0
                        values[missing] = -9999
                        dst.write(values, 1, window=block)
                dst.update_tags(dataset="GMIS 2010 v1", units="percent", source_archive=archive_path.name)
            staged.replace(normalized)
    normalized_files.append(normalized)
if not normalized_files:
    raise ValueError("No downloaded percentage tiles cover the DEM bounds.")
print("Source pixel size retained (degrees):", source_resolution)


## 5. Combine the tiles and mask the model area

Create one grid at the GMIS pixel size and keep the valid model area.


In [ ]:
with ExitStack() as stack:
    sources = [stack.enter_context(rasterio.open(path)) for path in normalized_files]
    mosaic, im_transform = merge(
        sources, bounds=dem_bounds, res=source_resolution,
        nodata=-9999, dtype="float32", indexes=[1],
        resampling=Resampling.nearest, target_aligned_pixels=True,
    )
    im_crs = sources[0].crs
image = mosaic[0]
with rasterio.open(dem_file) as dem:
    with WarpedVRT(dem, crs=im_crs, transform=im_transform,
                   width=image.shape[1], height=image.shape[0], dtype="float32",
                   nodata=-9999, resampling=Resampling.nearest) as template:
        dem_values = template.read(1, masked=True)
domain = ~np.ma.getmaskarray(dem_values) & np.isfinite(dem_values.data)
del dem_values
print("Impervious grid shape:", image.shape)
print("Impervious pixel size (degrees):", (im_transform.a, abs(im_transform.e)))


## 6. Fill missing cells

Missing cells inside the model area use nearby values. Inspect these estimates
in the filling mask. Set `fill_missing=False` to keep the gaps.


In [ ]:
valid = np.isfinite(image) & (image >= 0) & (image <= 100)
pending = domain & ~valid
filled = np.zeros(image.shape, dtype=bool)
print("Missing domain cells before filling:", int(pending.sum()))
if fill_missing and pending.any():
    donors = domain & valid
    if not donors.any():
        raise ValueError("No valid impervious percentage cells are available for nearest-neighbor filling.")
    nearest = distance_transform_edt(
        ~donors, sampling=(abs(im_transform.e), abs(im_transform.a)),
        return_distances=False, return_indices=True,
    )
    image[pending] = image[nearest[0][pending], nearest[1][pending]]
    filled[pending] = True
    del nearest, donors
image[~domain] = -9999
remaining = domain & (~np.isfinite(image) | (image < 0) | (image > 100))
image[remaining] = -9999
print("Filled domain cells:", int(filled.sum()))
print("Remaining missing domain cells:", int(remaining.sum()))


## 7. Save the grids

`crest_IM.tif` stores percentages, so 25 means 25% impervious surface.
The filling mask is saved separately in `posproc_outputs/crest_diagnostics`.
[Output codes and processing details](../../docs/impervious.md).


In [ ]:
for path in (output_file, fill_mask_file):
    if path.exists() and not overwrite:
        raise FileExistsError(f"{path} already exists; set overwrite=True to replace outputs.")
profile = dict(driver="GTiff", count=1, height=image.shape[0], width=image.shape[1],
               crs=im_crs, transform=im_transform, dtype="float32", nodata=-9999,
               tiled=False, interleave="pixel", compress="deflate", BIGTIFF="IF_SAFER")
with TemporaryDirectory(prefix=".gmis-output-", dir=output_dir) as temp:
    staged_im = Path(temp) / output_file.name
    staged_mask = Path(temp) / fill_mask_file.name
    with rasterio.open(staged_im, "w", **profile) as dst:
        dst.write(image, 1)
        dst.update_tags(dataset="GMIS 2010 v1", doi="10.7927/H4P55KKF", units="percent",
                        source_resolution="250m", reference_year="2010", resampling="nearest_mosaic_only",
                        filled_cells=str(int(filled.sum())), missing_cells=str(int(remaining.sum())))
    with rasterio.open(staged_mask, "w", **profile) as dst:
        dst.write(np.where(domain, filled.astype("float32"), -9999), 1)
        dst.update_tags(units="0=unchanged,1=nearest_filled", dataset="GMIS 2010 v1")
    staged_im.replace(output_file)
    staged_mask.replace(fill_mask_file)
print("Saved:", output_file.resolve())
print("Saved:", fill_mask_file.resolve())


## 8. Inspect and use the layer

Check the maps before adding `crest_IM.tif` to the CREST control-file block.
The saved raster keeps the source pixel size.


In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent

with rasterio.open(output_file) as src:
    scale = min(1.0, 1400/max(src.height, src.width))
    display = src.read(1, masked=True,
                       out_shape=(max(1, int(src.height*scale)), max(1, int(src.width*scale))),
                       resampling=Resampling.nearest)
    extent = plotting_extent(src)
    assert src.dtypes == ("float32",) and src.nodata == -9999
    assert not src.profile.get("tiled", False) and src.compression == rasterio.enums.Compression.deflate
    print("Output pixel size:", src.res, "CRS:", src.crs)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(display, extent=extent, origin="upper", cmap="viridis", vmin=0, vmax=100)
fig.colorbar(im, ax=ax, label="Impervious surface (%)")
ax.set(title="GMIS 2010 impervious surface — Ghana example", xlabel="Longitude", ylabel="Latitude")
plt.show()
print(f"im_grid={output_file.resolve()}")
